<h1><font color="#113D68" size=6>Deep Learning en Python con PyTorch</font></h1>

<h1><font color="#113D68" size=5>Parte 4. Mejorar la Generalización en modelos</font></h1>

<h1><font color="#113D68" size=4>1. Guardar y cargar modelos</font></h1>

<br><br>
<div style="text-align: right">
<font color="#113D68" size=3>Manuel Castillo Cara</font><br>

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
More information about [Manuel Castillo-Cara](https://www.manuelcastillo.eu/)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Puedes ver más cursos de Inteligencia Artificial, Machine Learning y Deep Learning en mi [página web](https://www.manuelcastillo.eu/udemy/)


---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>

* [0. Contexto](#section0)
* [1. Construcción de un Modelo](#section1)
* [2. ¿Qué hay dentro de un modelo de PyTorch?](#section2)
* [3. Acceder a `state_dict` de un Modelo](#section3)


---
<a id="section0"></a>
# <font color="#004D7F" size=6> 0. Contexto</font>

Un modelo de deep learning es una abstracción matemática de los datos, en la que se involucran muchos parámetros. Entrenar estos parámetros puede tomar horas, días o incluso semanas, pero después puedes usar el resultado para aplicarlo a nuevos datos. A esto se le llama inferencia en machine learning. Es importante saber cómo podemos preservar el modelo entrenado en el disco y, posteriormente, cargarlo para su uso en inferencia.

Después de leer este capítulo, sabrás:

- Qué son los estados y parámetros en un modelo de PyTorch.
- Cómo guardar los estados del modelo.
- Cómo cargar los estados del modelo.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
[PyTorch tutorials.](https://pytorch.org/tutorials/)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch API [`torch.nn`.](https://pytorch.org/docs/stable/nn.html)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch API [`torch.optim`.](https://pytorch.org/docs/stable/optim.html)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch Tutorials. [Guardar y cargar modelos](https://pytorch.org/docs/stable/optim.html)

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section1"></a>
# <font color="#004D7F" size=6>1. Construcción de un Modelo</font>

Empecemos con un modelo muy simple en PyTorch. 

- Es un modelo basado en el dataset de Iris, problema de clasificación multiclase. 

- En este modelo, se utiliza **log softmax**, i.e., `LogSoftmax()` como activación de salida.

- Por lo que se puede combinar con la función de pérdida de **verosimilitud negativa logarítmica**, i.e., `NLLLoss()`. 

- Esto es equivalente a no usar activación en la salida y combinarlo con la función de pérdida de entropía cruzada.

- Es por ese motivo que hacemos directamente un `LabelEncoder` a la salida `y` en vez de hacer un one-hot encoding.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre el dataset [Iris](https://archive.ics.uci.edu/ml/datasets/Iris)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre el dataset [`NLLLoss()`](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre el dataset [`LogSoftmax()`](https://pytorch.org/docs/stable/generated/torch.nn.LogSoftmax.html)

In [1]:
# Descarga de datos (necesario en Colab): se guardan en ./Datasets
import os, urllib.request

os.makedirs('Datasets', exist_ok=True)
urls = {
    'iris.csv':
        'https://raw.githubusercontent.com/jbrownlee/Datasets/master/iris.csv',
}
for nombre, url in urls.items():
    if not os.path.exists(f'Datasets/{nombre}'):
        urllib.request.urlretrieve(url, f'Datasets/{nombre}')

In [2]:
import torch
import torch.nn as nn
import pandas as pd
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

data = pd.read_csv("Datasets/iris.csv", header=None)
X = data.iloc[:, 0:4]
y = data.iloc[:, 4]

encoder = LabelEncoder()
encoder.fit(y)
y = encoder.transform(y)

X = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)  # índices de clase (NLLLoss los exige)

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True)

# Modelo PyTorch
class Multiclass(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(4, 8)
        self.act = nn.ReLU()
        self.output = nn.Linear(8, 3)
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = self.act(self.hidden(x))
        x = self.logsoftmax(self.output(x))
        return x
model = Multiclass()

loss_fn = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

n_epochs = 100
batch_size = 5
batch_start = torch.arange(0, len(X_train), batch_size)

for epoch in range(n_epochs):
    for start in batch_start:
        X_batch = X_train[start:start+batch_size]
        y_batch = y_train[start:start+batch_size]
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [3]:
y

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2])

Evaluamos en test para confirmar que funciona correctamente.

In [4]:
y_pred = model(X_test)
acc = (torch.argmax(y_pred, 1) == y_test).float().mean()
print("Accuracy: %.2f" % acc)

Accuracy: 1.00


---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section2"></a>
# <font color="#004D7F" size=6>2. ¿Qué hay dentro de un modelo de PyTorch?</font>

- Un modelo de **PyTorch** es un objeto en Python que contiene componentes como capas y funciones de activación.
  
- El modelo sabe cómo conectar estos componentes para generar una salida a partir de tensores de entrada.

- Aunque el algoritmo del modelo se define al crearlo, los **parámetros entrenables** se modifican durante el entrenamiento para mejorar la métrica.

- Los **parámetros del modelo** son registrados en el optimizador, lo que permite que sean ajustados durante el entrenamiento.

- Durante el bucle de entrenamiento, la optimización ocurre en tres pasos:

  1. **`optimizer.zero_grad()`**: Resetea los gradientes acumulados.

  2. **`loss.backward()`**: Calcula los gradientes mediante retropropagación.

  3. **`optimizer.step()`**: Actualiza los parámetros del modelo usando los gradientes calculados.

In [5]:
optimizer = optim.Adam(model.parameters(), lr=0.01)
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0
)

- La función `model.parameters()` proporciona un generador que hace referencia a los parámetros entrenables de cada capa en forma de tensores. 

- Por lo tanto, es posible hacer una copia de ellos o sobrescribirlos. 

A continuación se muestra un ejemplo de cómo copiar los pesos de un modelo a otro:

In [6]:
newmodel = Multiclass()

with torch.no_grad():
    for newparam, oldparam in zip(newmodel.parameters(), model.parameters()):
        newparam.copy_(oldparam)  # copia los pesos de un modelo al otro

y_pred = newmodel(X_test)
acc = (torch.argmax(y_pred, 1) == y_test).float().mean()
print("Accuracy: %.2f" % acc)

Accuracy: 1.00


- Al copiar los parámetros entre dos modelos, el resultado debería ser el mismo, ya que los modelos se vuelven idénticos en términos de sus parámetros entrenables.

- **Excepción**: Algunos modelos tienen **parámetros no entrenables**, como los de la capa **Batch Normalization**.

- **Batch Normalization**:
  - Aplica normalización a los tensores producidos por la capa anterior.

  - Pasa el tensor normalizado a la siguiente capa.
  
  - Tiene dos parámetros clave: **media** y **desviación estándar**.

- Estos parámetros (media y desviación estándar) se **aprenden** durante el entrenamiento a partir de los datos de entrada.

- Aunque son importantes, estos parámetros **no son entrenables por el optimizador** y, por lo tanto, **no forman parte de `model.parameters()`**.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section3"></a>
# <font color="#004D7F" size=6>3. Acceder a `state_dict` de un Modelo</font>

- Para acceder a **todos los parámetros** de un modelo (entrenables y no entrenables), se utiliza la función **`state_dict()`**.

- **`state_dict()`** contiene todas las **variables de estado** del modelo.

- El resultado es un **`OrderedDict`**, una estructura de datos del módulo `collections` en Python.

- Cada componente del modelo tiene un **nombre**, y los parámetros dentro de ellos también tienen nombres.

- El **`OrderedDict`** permite mapear correctamente los pesos a los parámetros correspondientes, coincidiendo sus nombres.

Vamos un ejemplo con `pprint`

In [7]:
import pprint
pp = pprint.PrettyPrinter()
pp.pprint(model.state_dict())

OrderedDict([('hidden.weight',
              tensor([[ 0.2571, -0.3854, -0.4543, -0.0758],
        [-0.4874,  0.2946, -0.0262, -0.3535],
        [-0.1316, -0.3679, -0.3270,  0.2822],
        [ 0.1676, -0.9219,  0.6403,  0.8406],
        [-0.2548, -0.4605,  1.3684,  1.2153],
        [-0.3225, -0.1446,  0.1052,  0.1523],
        [-0.4175, -0.4998,  0.4060, -0.3358],
        [ 0.7253,  0.9384, -0.9650, -1.7313]])),
             ('hidden.bias',
              tensor([-0.0059,  0.3567, -0.0709, -1.5049, -0.2226, -0.0684,  0.0316,  1.6006])),
             ('output.weight',
              tensor([[ 0.2362,  0.3443, -0.0829, -0.6303, -2.5971,  0.1817,  0.0701,  2.2653],
        [ 0.0554,  0.1216,  0.2078, -0.4737,  0.1634, -0.0090,  0.2231,  1.0599],
        [-0.2972,  0.1093,  0.0722,  0.4829,  0.8083,  0.0628, -0.0860, -2.8570]])),
             ('output.bias', tensor([ 0.1367,  0.3134, -0.4847]))])


Para la inferencia, primero creas un modelo (sin entrenarlo) y luego cargas los estados. 

En Python, el formato nativo para la serialización es `pickle`:

In [8]:
import pickle

with open('iris-model.pickle', 'wb') as fp:
    pickle.dump(model.state_dict(), fp)

newmodel = Multiclass()  # primero se recrea el modelo (sin entrenar)
with open('iris-model.pickle', 'rb') as fp:
    newmodel.load_state_dict(pickle.load(fp))

y_pred = newmodel(X_test)
acc = (torch.argmax(y_pred, 1) == y_test).float().mean()
print("Accuracy: %.2f" % acc)

Accuracy: 1.00


In [9]:
newmodel

Multiclass(
  (hidden): Linear(in_features=4, out_features=8, bias=True)
  (act): ReLU()
  (output): Linear(in_features=8, out_features=3, bias=True)
  (logsoftmax): LogSoftmax(dim=1)
)

Se recomienda utilizar la API de PyTorch para guardar y cargar los estados en lugar de usar `pickle` manualmente:

In [10]:
torch.save(model.state_dict(), 'iris-model.pth')

newmodel = Multiclass()
newmodel.load_state_dict(torch.load('iris-model.pth'))

y_pred = newmodel(X_test)
acc = (torch.argmax(y_pred, 1) == y_test).float().mean()
print("Accuracy: %.2f" % acc)

Accuracy: 1.00


In [11]:
newmodel

Multiclass(
  (hidden): Linear(in_features=4, out_features=8, bias=True)
  (act): ReLU()
  (output): Linear(in_features=8, out_features=3, bias=True)
  (logsoftmax): LogSoftmax(dim=1)
)

- El archivo **`*.pth`** es un archivo comprimido que contiene varios archivos **`pickle`** creados por PyTorch.

- **Recomendación**: Usar este formato porque PyTorch puede almacenar **información adicional** en él.

- Al guardar solo los **estados** del modelo, no estás guardando el modelo completo, solo sus parámetros.

- Para usar los estados guardados, necesitas:
  - **Recrear el modelo** en Python.
  - Cargar los estados guardados en el modelo recreado.

- Si deseas guardar también el **modelo completo**, puedes hacerlo pasando el modelo entero en lugar de solo los estados.

Veamos un ejemplo de guardado y carga completo.

> **Actualización:** desde PyTorch 2.6 `torch.load` usa `weights_only=True` por defecto. Eso es suficiente para cargar un `state_dict`, pero para cargar un **modelo completo** hay que indicar `weights_only=False` (solo con archivos de confianza, porque usa `pickle`).

In [12]:
torch.save(model, 'iris-model-full.pth')

newmodel = torch.load('iris-model-full.pth', weights_only=False)

y_pred = newmodel(X_test)
acc = (torch.argmax(y_pred, 1) == y_test).float().mean()
print("Accuracy: %.2f" % acc)

Accuracy: 1.00


In [13]:
newmodel

Multiclass(
  (hidden): Linear(in_features=4, out_features=8, bias=True)
  (act): ReLU()
  (output): Linear(in_features=8, out_features=3, bias=True)
  (logsoftmax): LogSoftmax(dim=1)
)

In [14]:
newmodel1 = Multiclass()
newmodel1.load_state_dict(torch.load('iris-model.pth'))
newmodel1

Multiclass(
  (hidden): Linear(in_features=4, out_features=8, bias=True)
  (act): ReLU()
  (output): Linear(in_features=8, out_features=3, bias=True)
  (logsoftmax): LogSoftmax(dim=1)
)

In [15]:
newmodel1 = torch.load('iris-model-full.pth', weights_only=False)
newmodel1

Multiclass(
  (hidden): Linear(in_features=4, out_features=8, bias=True)
  (act): ReLU()
  (output): Linear(in_features=8, out_features=3, bias=True)
  (logsoftmax): LogSoftmax(dim=1)
)

<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>